# Modeling for analytics: star schemas, views and metric views

Analytics models organise data so questions are easy and answers are consistent. A star schema keeps measurable events in a fact table and descriptive attributes in dimension tables. Views save a query under a name; standalone materialized views store and refresh a query's result. Unity Catalog metric views define measures once, separate from dimensions, so every dashboard, Genie Agent and query aggregates them the same way, at whatever grain the user picks.

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end.

From the Learning Hub lesson: [Modeling for analytics: star schemas, views and metric views](https://www.datapreplab.com/hub#topic/analytics-modeling) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** build a tiny star, see the average-of-averages trap, and fix it with a metric view.

**Part A: the star.**

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.dim_customer_lab AS
SELECT * FROM VALUES (1, 'Enterprise'), (2, 'SMB'), (3, 'SMB') AS t(customer_id, segment);

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.fact_orders_lab AS
SELECT * FROM VALUES
  (101, 1, DATE'2026-10-01', 900.00), (102, 2, DATE'2026-10-01', 50.00),
  (103, 3, DATE'2026-10-02', 30.00),  (104, 2, DATE'2026-10-02', 20.00),
  (105, 3, DATE'2026-10-02', 40.00)
AS t(order_id, customer_id, order_date, amount);

**Part B: the trap.** Predict both numbers, then run:

In [ ]:
%sql
WITH daily AS (
  SELECT order_date, avg(amount) AS day_avg FROM workspace.default.fact_orders_lab GROUP BY order_date)
SELECT (SELECT avg(day_avg) FROM daily) AS average_of_daily_averages,
       (SELECT sum(amount) / count(*) FROM workspace.default.fact_orders_lab) AS true_average;

**Part C: the metric view.**

In [ ]:
%sql
CREATE OR REPLACE VIEW workspace.default.sales_metrics_lab
WITH METRICS LANGUAGE YAML AS $$
version: 1.1
source: workspace.default.fact_orders_lab
joins:
  - name: customer
    source: workspace.default.dim_customer_lab
    on: source.customer_id = customer.customer_id
dimensions:
  - name: Order Date
    expr: source.order_date
  - name: Segment
    expr: customer.segment
measures:
  - name: Revenue
    expr: SUM(source.amount)
  - name: Orders
    expr: COUNT(1)
  - name: Average Order Value
    expr: SUM(source.amount) / COUNT(1)
$$;

**Part D: query at different grains.**

In [ ]:
%sql
SELECT Segment, MEASURE(Revenue) AS revenue, MEASURE(`Average Order Value`) AS aov
FROM workspace.default.sales_metrics_lab
GROUP BY ALL ORDER BY Segment;

In [ ]:
%sql
SELECT MEASURE(`Average Order Value`) AS overall_aov
FROM workspace.default.sales_metrics_lab;

Expected evidence:

```text
Part B -> average of daily averages 252.50 (475 and 30) ; true average 208.00
Part D -> Enterprise revenue 900.00 aov 900.00 ; SMB revenue 140.00 aov 35.00 ; overall aov 208.00 (shown with many decimal places; wrap it in round(..., 2) for display)
SELECT * -> fails with METRIC_VIEW_MISSING_MEASURE_FUNCTION: measures need MEASURE()
```

**Cleanup:**

In [ ]:
%sql
DROP VIEW workspace.default.sales_metrics_lab;

In [ ]:
%sql
DROP TABLE workspace.default.fact_orders_lab;

In [ ]:
%sql
DROP TABLE workspace.default.dim_customer_lab;